# Resampling-Time Tables

Generate paper-ready timing tables from all ten completed `with_c` timing batches. This notebook validates saved outputs, never reruns models, and overwrites `latex/tables/timing_results.tex`.

## Load and validate timing outputs

The timing evidence must contain ten batches, 30 datasets, 100 folds per dataset-method pair, Re-SC and KMeans-ReSC only, non-negative timings, and internal time no greater than end-to-end sampler time.

In [3]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd


def find_experiment_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'results' / 'metrics').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Run notebook from resc-experiments or one of its subdirectories.')


ROOT = find_experiment_root()
METRICS_DIR = ROOT / 'results' / 'metrics'
LATEX_TABLE_PATH = ROOT.parent / 'latex' / 'tables' / 'timing_results.tex'
MANIFEST_PATH = METRICS_DIR / 'dataset_manifest.json'
RUN_PREFIX = 'timing_fixed_m15_with_c_batch'
EXPECTED_BATCHES = 10
EXPECTED_DATASETS = 30
EXPECTED_FOLDS = 100
EXPECTED_METHODS = {'resc', 'kmeans_resc_q10'}

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
dataset_labels = {entry['dataset_key']: entry['source_name'] for entry in manifest['datasets']}

batch_paths = [
    METRICS_DIR / f'{RUN_PREFIX}{batch:02d}of{EXPECTED_BATCHES}' / 'with_c' / 'phase_timings_outer_folds.csv'
    for batch in range(1, EXPECTED_BATCHES + 1)
]
missing = [str(path) for path in batch_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing timing batch output(s):\n' + '\n'.join(missing))

batch_frames = [pd.read_csv(path) for path in batch_paths]
for batch_number, frame in enumerate(batch_frames, start=1):
    if len(frame) != 600:
        raise ValueError(f'Batch {batch_number} has {len(frame)} rows; expected 600.')

timings = pd.concat(batch_frames, ignore_index=True)
if len(timings) != EXPECTED_DATASETS * EXPECTED_FOLDS * len(EXPECTED_METHODS):
    raise ValueError(f'Expected 6000 timing rows; found {len(timings)}.')
if set(timings['track']) != {'with_c'}:
    raise ValueError('Timing batches must contain only the with_c track.')
if set(timings['method']) != EXPECTED_METHODS:
    raise ValueError(f'Unexpected timing methods: {sorted(timings["method"].unique())}')

fold_keys = ['dataset_key', 'track', 'repeat', 'fold', 'method']
if timings.duplicated(fold_keys).any():
    raise ValueError('Duplicate dataset-fold-method timing rows found.')
fold_counts = timings.groupby(['dataset_key', 'method']).size()
if timings['dataset_key'].nunique() != EXPECTED_DATASETS or not fold_counts.eq(EXPECTED_FOLDS).all():
    raise ValueError('Each dataset-method pair must contain exactly 100 folds.')

time_columns = [column for column in timings.columns if column.endswith('_seconds')]
if (timings[time_columns].fillna(0.0) < 0.0).any().any():
    raise ValueError('Negative timing found.')
if not (timings['total_internal_seconds'] <= timings['end_to_end_resampling_seconds'] + 1e-9).all():
    raise ValueError('Internal resampling time exceeds end-to-end sampler time.')

display(fold_counts.unstack('method'))
print(f'Validated {len(timings)} timing rows from {EXPECTED_BATCHES} batches and {EXPECTED_DATASETS} datasets.')

method,kmeans_resc_q10,resc
dataset_key,,
imblearn_arrhythmia,100,100
imblearn_car_eval_34,100,100
imblearn_ecoli,100,100
imblearn_libras_move,100,100
imblearn_oil,100,100
imblearn_solar_flare_m0,100,100
imblearn_spectrometer,100,100
imblearn_us_crime,100,100
imblearn_yeast_me2,100,100


Validated 6000 timing rows from 10 batches and 30 datasets.


## Render and write LaTeX tables

All durations are reported in milliseconds. Generated tables: overall timing and representative counts, internal phase decomposition, and a 30-dataset timing comparison with candidate/fallback diagnostics.

In [4]:
METHOD_LABELS = {'resc': 'Re-SC', 'kmeans_resc_q10': 'KMeans-ReSC'}
PHASE_LABELS = {
    'validation_seconds': 'Validation',
    'representative_size_seconds': 'Representative-size calculation',
    'normalization_seconds': 'Normalization',
    'neighbor_fit_seconds': 'Neighbor fit',
    'neighbor_query_seconds': 'Neighbor query',
    'weight_calculation_seconds': 'Weight calculation',
    'weighted_sampling_seconds': 'Weighted sampling',
    'representative_bounds_seconds': 'Representative bounds',
    'safety_neighbor_fit_seconds': 'Safety-neighbor fit',
    'safety_neighbor_query_seconds': 'Safety-neighbor query',
    'safety_filter_seconds': 'Safety filtering',
    'candidate_grid_seconds': 'Candidate-grid construction',
    'candidate_kmeans_fit_seconds': 'Candidate KMeans fits',
    'silhouette_score_seconds': 'Silhouette scoring',
    'final_kmeans_fit_seconds': 'Final KMeans fit',
    'concatenation_seconds': 'Concatenation',
    'unattributed_seconds': 'Other internal operations',
}
RE_SC_PHASES = [
    'validation_seconds', 'representative_size_seconds', 'normalization_seconds',
    'neighbor_fit_seconds', 'neighbor_query_seconds', 'weight_calculation_seconds',
    'weighted_sampling_seconds', 'concatenation_seconds', 'unattributed_seconds',
]
KMEANS_RESC_PHASES = [
    'validation_seconds', 'representative_bounds_seconds', 'safety_neighbor_fit_seconds',
    'safety_neighbor_query_seconds', 'safety_filter_seconds', 'candidate_grid_seconds',
    'candidate_kmeans_fit_seconds', 'silhouette_score_seconds', 'final_kmeans_fit_seconds',
    'concatenation_seconds', 'unattributed_seconds',
]


def latex_escape(value: object) -> str:
    return str(value).replace('\\', r'\textbackslash{}').replace('_', r'\_').replace('&', r'\&')


def milliseconds(value: float) -> float:
    return 1000.0 * float(value)


def wrap_table(frame: pd.DataFrame, caption: str, label: str, column_format: str, environment: str = 'table', font: str = r'\scriptsize', tabcolsep: str = '2.0pt') -> str:
    tabular = frame.to_latex(index=False, escape=False, column_format=column_format)
    return (
        f'\\begin{{{environment}}}[!t]\n'
        f'\\centering\n'
        f'\\caption{{{caption}}}\n'
        f'\\label{{{label}}}\n'
        f'\\setlength{{\\tabcolsep}}{{{tabcolsep}}}\n'
        f'\\renewcommand{{\\arraystretch}}{{0.86}}\n'
        f'{font}\n'
        f'{tabular}\n'
        f'\\end{{{environment}}}'
    )


overall = timings.groupby('method', sort=False).agg(
    mean_seconds=('end_to_end_resampling_seconds', 'mean'),
    median_seconds=('end_to_end_resampling_seconds', 'median'),
    p95_seconds=('end_to_end_resampling_seconds', lambda values: values.quantile(0.95)),
    mean_representatives=('representative_count', 'mean'),
    median_representatives=('representative_count', 'median'),
).reindex(['resc', 'kmeans_resc_q10']).reset_index()
overall_table = pd.DataFrame({
    'Method': overall['method'].map(METHOD_LABELS),
    'Mean (ms)': overall['mean_seconds'].map(milliseconds).map(lambda value: f'{value:.2f}'),
    'Median (ms)': overall['median_seconds'].map(milliseconds).map(lambda value: f'{value:.2f}'),
    'P95 (ms)': overall['p95_seconds'].map(milliseconds).map(lambda value: f'{value:.2f}'),
    'Mean reps.': overall['mean_representatives'].map(lambda value: f'{value:.1f}'),
    'Median reps.': overall['median_representatives'].map(lambda value: f'{value:.1f}'),
})
overall_tex = wrap_table(
    overall_table,
    r'Fold-level resampling time and representative-set size across 30 datasets. Times are wall-clock durations of \texttt{fit\_resample}, excluding preprocessing and classifier fitting.',
    'tab:timing_overall',
    '@{}lrrrrr@{}',
    environment='table*',
    tabcolsep='1.2pt',
)

phase_records = []
for method, phases in [('resc', RE_SC_PHASES), ('kmeans_resc_q10', KMEANS_RESC_PHASES)]:
    method_rows = timings.loc[timings['method'].eq(method)]
    for phase in phases:
        values = method_rows[phase].fillna(0.0)
        shares = 100.0 * values / method_rows['total_internal_seconds']
        phase_records.append({
            'Method': METHOD_LABELS[method],
            'Phase': PHASE_LABELS[phase],
            'Mean (ms)': f'{milliseconds(values.mean()):.2f}',
            'Median (ms)': f'{milliseconds(values.median()):.2f}',
            r'Share (\%)': f'{shares.mean():.1f}',
        })
phase_table = pd.DataFrame(phase_records)
phase_tabular = phase_table.to_latex(index=False, escape=False, column_format='@{}llrrr@{}')
phase_pair_tex = (
    r'\begin{figure*}[!t]' '\n'
    r'\begin{minipage}[t]{0.48\textwidth}' '\n'
    r'\vspace{0pt}' '\n'
    r'\centering' '\n'
    r'\includegraphics[width=\linewidth]{figures/imbalance_ratio_before_after.pdf}' '\n'
    r'\captionof{figure}{Mean directional imbalance ratio before resampling and after Re-SC and KMeans-ReSC across 100 training folds per dataset. Markers sharing a horizontal coordinate represent the same dataset, and connectors show the difference between resamplers. Both axes are logarithmic; the dashed horizontal line marks post-resampling class parity at $IR=1$. Values below one indicate reversal of the original class-size ordering.}' '\n'
    r'\label{fig:imbalance_ratio_before_after}' '\n'
    r'\end{minipage}' '\n'
    r'\hfill' '\n'
    r'\begin{minipage}[t]{0.48\textwidth}' '\n'
    r'\vspace{0pt}' '\n'
    r'\centering' '\n'
    r'\captionof{table}{Internal resampling-time decomposition. Shares are mean percentages of total internal sampler time over 3,000 folds per method.}' '\n'
    r'\label{tab:timing_phase_breakdown}' '\n'
    r'\setlength{\tabcolsep}{2.0pt}' '\n'
    r'\renewcommand{\arraystretch}{0.86}' '\n'
    r'\scriptsize' '\n'
    f'{phase_tabular}\n'
    r'\end{minipage}' '\n'
    r'\end{figure*}'
)
tradeoff_tex = (
    r'\begin{figure*}[!t]' '\n'
    r'\centering' '\n'
    r'\includegraphics[width=0.82\textwidth]{figures/resampling_time_representative_tradeoff.pdf}' '\n'
    r'\caption{Dataset-level trade-off between resampling time and representative-set size. Each point represents one dataset. The horizontal coordinate is the ratio of KMeans-ReSC to Re-SC mean resampling time on a logarithmic scale; the vertical coordinate is the corresponding representative-count ratio. Dashed reference lines at one mark equal time and equal representative count. Labels identify isolated extremes only.}' '\n'
    r'\label{fig:resampling_time_representative_tradeoff}' '\n'
    r'\end{figure*}'
)

dataset_order = list(dict.fromkeys(timings['dataset_key']))
dataset_table = timings.groupby(['dataset_key', 'method'], sort=False).agg(
    resc_seconds=('end_to_end_resampling_seconds', 'mean'),
    candidates=('actual_k_candidates', 'mean'),
    selected_k=('selected_k', 'mean'),
    selection_fallback=('selection_fallback_used', 'mean'),
).reset_index().pivot(index='dataset_key', columns='method').reindex(dataset_order)
per_dataset_table = pd.DataFrame({
    'Dataset': [latex_escape(dataset_labels[dataset]) for dataset in dataset_table.index],
    'Re-SC (ms)': dataset_table[('resc_seconds', 'resc')].map(milliseconds).map(lambda value: f'{value:.2f}'),
    'KMeans-ReSC (ms)': dataset_table[('resc_seconds', 'kmeans_resc_q10')].map(milliseconds).map(lambda value: f'{value:.2f}'),
    'Ratio': (dataset_table[('resc_seconds', 'kmeans_resc_q10')] / dataset_table[('resc_seconds', 'resc')]).map(lambda value: f'{value:.2f}'),
    'Candidates': dataset_table[('candidates', 'kmeans_resc_q10')].map(lambda value: f'{value:.1f}'),
    'Selected $K$': dataset_table[('selected_k', 'kmeans_resc_q10')].map(lambda value: f'{value:.1f}'),
    r'Selection fallback (\%)': (100.0 * dataset_table[('selection_fallback', 'kmeans_resc_q10')]).map(lambda value: f'{value:.0f}'),
})
dataset_tex = wrap_table(
    per_dataset_table,
    r'Mean fold-level resampling time by dataset. Ratio is KMeans-ReSC time divided by Re-SC time; values above one indicate greater KMeans-ReSC cost. Candidate count, selected $K$, and selection-fallback rate describe KMeans-ReSC selection.',
    'tab:timing_by_dataset',
    'lrrrrrr',
    environment='table*',
    font=r'\fontsize{6.5}{7.5}\selectfont',
    tabcolsep='1.2pt',
)

tex_content = '\n\n'.join([overall_tex, phase_pair_tex, tradeoff_tex, dataset_tex]) + '\n'
LATEX_TABLE_PATH.write_text(tex_content, encoding='utf-8')
print(f'Wrote {LATEX_TABLE_PATH}')
display(overall_table)
display(phase_table)
display(per_dataset_table)

Wrote C:\Users\Maxim\Desktop\re-sc\latex\tables\timing_results.tex


,Method,Mean (ms),Median (ms),P95 (ms),Mean representatives,Median representatives
0,Re-SC,10.23,5.87,23.76,30.1,16.5
1,KMeans-ReSC,175.74,86.32,733.88,23.9,12.0


,Method,Phase,Mean (ms),Median (ms),Share (\%)
0,Re-SC,Validation,0.06,0.04,1.0
1,Re-SC,Representative-size calculation,0.17,0.15,3.4
2,Re-SC,Normalization,0.10,0.05,1.3
3,Re-SC,Neighbor fit,0.29,0.24,5.9
4,Re-SC,Neighbor query,2.75,0.92,26.5
5,Re-SC,Weight calculation,1.77,1.27,23.6
6,Re-SC,Weighted sampling,0.23,0.21,5.0
7,Re-SC,Concatenation,3.98,1.08,29.1
8,Re-SC,Other internal operations,0.21,0.18,4.2
9,KMeans-ReSC,Validation,0.02,0.02,0.1


,Dataset,Re-SC (ms),KMeans-ReSC (ms),Ratio,Candidates,Selected $K$,Selection fallback (\%)
dataset_key,,,,,,,
imblearn_arrhythmia,arrhythmia,9.75,9.42,0.97,0.0,1.0,100
imblearn_car_eval_34,car\_eval\_34,20.91,259.61,12.42,6.0,13.2,0
imblearn_ecoli,ecoli,3.14,16.42,5.23,2.0,4.0,0
imblearn_libras_move,libras\_move,4.85,5.00,1.03,0.0,1.0,100
imblearn_oil,oil,9.00,8.89,0.99,0.0,1.0,100
imblearn_solar_flare_m0,solar\_flare\_m0,17.67,68.32,3.87,2.0,4.0,0
imblearn_spectrometer,spectrometer,11.26,49.07,4.36,2.0,3.8,0
imblearn_us_crime,us\_crime,70.49,811.99,11.52,6.0,10.4,0
imblearn_yeast_me2,yeast\_me2,19.39,18.86,0.97,0.0,1.0,100


## Include in the paper

After both code cells run successfully, add `\input{tables/timing_results.tex}` near the computational-results discussion in `main.tex`. References: `tab:timing_overall`, `tab:timing_phase_breakdown`, and `tab:timing_by_dataset`.